# API to Parquet

In [ ]:
import logging
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# CONFIG

CITY_NAME = "Faisalabad"
LATITUDE = 31.4187
LONGITUDE = 73.0791
PARQUET_FILE_PATH = "faisalabad_aqi_2years.parquet"

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
MAX_RETRIES = 4
RETRY_BACKOFF_SECONDS = 5
REQUEST_TIMEOUT = 30
BURNING_SEASON_MONTHS = {10, 11}



# 1. FETCHING (with retries + response validation)

def _get_with_retries(url: str, params: dict) -> dict:
    """GET with exponential backoff. Open-Meteo returns HTTP 200 even for
    invalid parameter combos, so we also check for an 'error' key in the
    payload rather than trusting the status code alone."""
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            data = resp.json()
            if isinstance(data, dict) and data.get("error"):
                raise ValueError(f"API returned error payload: {data.get('reason')}")
            return data
        except (requests.RequestException, ValueError) as exc:
            last_exc = exc
            wait = RETRY_BACKOFF_SECONDS * attempt
            log.warning(
                "Request to %s failed (attempt %d/%d): %s — retrying in %ds",
                url, attempt, MAX_RETRIES, exc, wait,
            )
            time.sleep(wait)
    raise RuntimeError(f"Giving up on {url} after {MAX_RETRIES} attempts") from last_exc


def fetch_weather_and_air_quality_chunk(start_date: str, end_date: str) -> pd.DataFrame:
    """Fetches weather and air quality for a specific date range."""
    weather_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "surface_pressure",
            "wind_speed_10m",
            "wind_direction_10m",
        ],
        "timezone": "UTC",
    }
    aq_params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ["pm2_5", "pm10", "nitrogen_dioxide", "ozone", "us_aqi"],
        "timezone": "UTC",
    }

    log.info("Fetching data from %s to %s...", start_date, end_date)
    res_weather = _get_with_retries(WEATHER_URL, weather_params)
    res_aq = _get_with_retries(AQ_URL, aq_params)

    df_weather = pd.DataFrame(res_weather["hourly"])
    df_weather["time"] = pd.to_datetime(df_weather["time"])

    df_aq = pd.DataFrame(res_aq["hourly"])
    df_aq["time"] = pd.to_datetime(df_aq["time"])

    df = pd.merge(df_weather, df_aq, on="time", how="inner")
    df["city"] = CITY_NAME
    return df



# 2. GAP HANDLING

def fill_small_gaps(df: pd.DataFrame, max_gap_hours: int = 3) -> pd.DataFrame:
    """Reindexes to a complete hourly timeline and interpolates gaps up to
    max_gap_hours. Open-Meteo occasionally has missing hours for a given
    station/model; leaving raw NaNs would silently corrupt lag/rolling
    features far beyond the actual gap."""
    df = df.set_index("time").sort_index()
    full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
    n_missing = len(full_index) - len(df)
    if n_missing > 0:
        log.info("Reindexing: %d missing hourly timestamps found, interpolating short gaps", n_missing)
    df = df.reindex(full_index)
    df["city"] = df["city"].ffill()  # constant column, safe to forward-fill
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].interpolate(method="linear", limit=max_gap_hours)
    df = df.reset_index().rename(columns={"index": "time"})
    return df



# 3. FEATURE ENGINEERING

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Computes the full feature set. One row per hourly timestamp; targets
    are the ACTUAL future AQI values (direct multi-horizon forecasting)."""
    df = df.sort_values("time").reset_index(drop=True)

    # 1. Wind vector components (better ML signal than raw speed+direction)
    wind_rad = np.radians(df["wind_direction_10m"])
    df["wind_u"] = -df["wind_speed_10m"] * np.sin(wind_rad)
    df["wind_v"] = -df["wind_speed_10m"] * np.cos(wind_rad)

    # 2. Temporal & cyclical features
    df["hour"] = df["time"].dt.hour.astype("int64")
    df["dayofweek"] = df["time"].dt.dayofweek.astype("int64")
    df["month"] = df["time"].dt.month.astype("int64")
    df["dayofyear"] = df["time"].dt.dayofyear.astype("int64")
    df["is_weekend"] = (df["dayofweek"] >= 5).astype("int64")
    df["is_burning_season"] = df["month"].isin(BURNING_SEASON_MONTHS).astype("int64")

    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24.0)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12.0)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12.0)

    # 3. Lag features — pollution
    df["pm2_5_lag_1h"] = df["pm2_5"].shift(1)
    df["pm2_5_lag_24h"] = df["pm2_5"].shift(24)
    df["us_aqi_lag_1h"] = df["us_aqi"].shift(1)
    df["us_aqi_lag_24h"] = df["us_aqi"].shift(24)
    df["us_aqi_lag_48h"] = df["us_aqi"].shift(48)
    df["us_aqi_lag_72h"] = df["us_aqi"].shift(72)

    # 3b. Lag features — weather (added: weather trend is predictive too,
    # not just current-instant weather)
    df["wind_speed_lag_24h"] = df["wind_speed_10m"].shift(24)
    df["humidity_lag_24h"] = df["relative_humidity_2m"].shift(24)

    # 4. Rolling features — mean AND volatility (std/min/max), shift(1) first
    # in every case so the current row never sees its own value.
    df["pm2_5_roll_mean_6h"] = df["pm2_5"].shift(1).rolling(6).mean()
    df["pm2_5_roll_mean_24h"] = df["pm2_5"].shift(1).rolling(24).mean()
    df["pm2_5_roll_std_24h"] = df["pm2_5"].shift(1).rolling(24).std()
    df["us_aqi_roll_mean_24h"] = df["us_aqi"].shift(1).rolling(24).mean()
    df["us_aqi_roll_std_24h"] = df["us_aqi"].shift(1).rolling(24).std()
    df["us_aqi_roll_min_24h"] = df["us_aqi"].shift(1).rolling(24).min()
    df["us_aqi_roll_max_24h"] = df["us_aqi"].shift(1).rolling(24).max()

    # 5. Rates of change
    df["aqi_change_rate_1h"] = (df["pm2_5_lag_1h"] - df["pm2_5"].shift(2)) / (
        df["pm2_5"].shift(2) + 1e-5
    )
    df["aqi_change_rate_24h"] = (df["pm2_5_lag_1h"] - df["pm2_5_lag_24h"]) / (
        df["pm2_5_lag_24h"] + 1e-5
    )

    # 6. Pollutant composition ratio (dust vs. combustion signature)
    df["pm25_pm10_ratio"] = df["pm2_5"] / (df["pm10"] + 1e-5)

    # 7. Target horizons — direct multi-horizon targets, one column each
    df["target_aqi_24h"] = df["us_aqi"].shift(-24)
    df["target_aqi_48h"] = df["us_aqi"].shift(-48)
    df["target_aqi_72h"] = df["us_aqi"].shift(-72)

    before = len(df)
    df = df.dropna().reset_index(drop=True)
    log.info("Dropped %d rows with incomplete lag/target windows (%d remain)", before - len(df), len(df))

    # Datetime/type casting for Hopsworks compatibility
    df["city"] = df["city"].astype(str)
    df["us_aqi"] = df["us_aqi"].astype("int64")
    df["time"] = (
        pd.to_datetime(df["time"])
        .dt.tz_localize("UTC")
        .dt.tz_localize(None)
        .astype("datetime64[us]")
    )

    return df




def main():
    today = datetime.now()
    two_years_ago = today - timedelta(days=730)
    midpoint = two_years_ago + timedelta(days=365)

    chunks = [
        (two_years_ago.strftime("%Y-%m-%d"), midpoint.strftime("%Y-%m-%d")),
        ((midpoint + timedelta(days=1)).strftime("%Y-%m-%d"), today.strftime("%Y-%m-%d")),
    ]

    raw_dfs = []
    for start_date, end_date in chunks:
        df_chunk = fetch_weather_and_air_quality_chunk(start_date, end_date)
        raw_dfs.append(df_chunk)
        time.sleep(1)  # polite spacing, well within Open-Meteo's free-tier limits

    full_raw_df = pd.concat(raw_dfs, ignore_index=True).drop_duplicates(subset="time")
    full_raw_df = fill_small_gaps(full_raw_df)

    log.info("Engineering features across %d raw hourly rows...", len(full_raw_df))
    features_df = engineer_features(full_raw_df)

    features_df.to_parquet(PARQUET_FILE_PATH, index=False)
    log.info(
        "Successfully created %s with %d rows and %d columns.",
        PARQUET_FILE_PATH, len(features_df), len(features_df.columns),
    )


if __name__ == "__main__":
    main()

2026-08-21 10:59:07,637 [INFO] Fetching data from 2024-08-21 to 2025-08-21...
2026-08-21 10:59:40,654 [WARNING] Request to https://archive-api.open-meteo.com/v1/archive failed (attempt 1/4): HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=30) — retrying in 5s
2026-08-21 11:00:17,036 [WARNING] Request to https://archive-api.open-meteo.com/v1/archive failed (attempt 2/4): HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=30) — retrying in 10s
2026-08-21 11:01:36,753 [INFO] Fetching data from 2025-08-22 to 2026-08-21...
2026-08-21 11:02:11,407 [INFO] Engineering features across 17544 raw hourly rows...
2026-08-21 11:02:11,451 [INFO] Dropped 144 rows with incomplete lag/target windows (17400 remain)
2026-08-21 11:02:11,766 [INFO] Successfully created faisalabad_aqi_2years.parquet with 17400 rows and 45 columns.


In [ ]:
"""
Reads the Parquet file produced by fetch_and_engineer.py and ingests it into
the Hopsworks feature store.

Requires: pip install hopsworks
Env var:  HOPSWORKS_API_KEY

Run fetch_and_engineer.py first to produce the Parquet file this script reads.
"""

import logging
import os
import time

import pandas as pd
import hopsworks

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# CONFIG

PROJECT_NAME = "AQI_Predictor_fsd"
HOPSWORKS_HOST = "eu-west.cloud.hopsworks.ai"
HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")

PARQUET_FILE_PATH = "faisalabad_aqi_2years.parquet"

FEATURE_GROUP_NAME = "weather_aqi_hourly"
FEATURE_GROUP_VERSION = 5  
PRIMARY_KEY = ["city", "time"]
EVENT_TIME = "time"
TIME_TRAVEL_FORMAT = "HUDI"
INSERT_CHUNK_SIZE = 3000
MAX_INSERT_RETRIES = 3
RETRY_BACKOFF_SECONDS = 10

FEATURE_DESCRIPTIONS = {
    "city": "City the reading belongs to",
    "time": "Hourly timestamp (UTC), event-time / primary key",
    "us_aqi": "US AQI at this timestamp",
    "pm2_5": "PM2.5 concentration (µg/m³)",
    "pm10": "PM10 concentration (µg/m³)",
    "pm25_pm10_ratio": "PM2.5/PM10 ratio — combustion vs. dust signature",
    "is_burning_season": "1 if month is Oct/Nov (Punjab crop-residue burning season)",
    "target_aqi_24h": "Actual US AQI 24h ahead — regression target",
    "target_aqi_48h": "Actual US AQI 48h ahead — regression target",
    "target_aqi_72h": "Actual US AQI 72h ahead — regression target",
}


def load_features() -> pd.DataFrame:
    if not os.path.exists(PARQUET_FILE_PATH):
        raise FileNotFoundError(
            f"{PARQUET_FILE_PATH} not found — run fetch_and_engineer.py first."
        )
    df = pd.read_parquet(PARQUET_FILE_PATH)
    log.info("Loaded %d rows, %d columns from %s", len(df), len(df.columns), PARQUET_FILE_PATH)
    return df


def connect_feature_store():
    if not HOPSWORKS_API_KEY:
        raise ValueError("HOPSWORKS_API_KEY environment variable is not set!")

    project = hopsworks.login(
        host=HOPSWORKS_HOST,
        port=443,
        project=PROJECT_NAME,
        api_key_value=HOPSWORKS_API_KEY,
    )
    return project.get_feature_store()


def get_or_create_fg(fs):
    fg = fs.get_or_create_feature_group(
        name=FEATURE_GROUP_NAME,
        version=FEATURE_GROUP_VERSION,
        primary_key=PRIMARY_KEY,
        event_time=EVENT_TIME,
        time_travel_format=TIME_TRAVEL_FORMAT,
        description=(
            "Hourly weather, air quality, and engineered lag/rolling/calendar "
            "features for Faisalabad, with 24h/48h/72h AQI forecast targets."
        ),
        online_enabled=True,
    )

    log.info("Feature group '%s' v%d resolved time_travel_format=%s",
              fg.name, fg.version, fg.time_travel_format)
    if fg.time_travel_format != TIME_TRAVEL_FORMAT:
        raise RuntimeError(
            f"Requested time_travel_format='{TIME_TRAVEL_FORMAT}' but the feature "
            f"group resolved to '{fg.time_travel_format}'. Writing DELTA from an "
            f"external client requires direct HDFS RPC access, which is almost "
            f"never reachable off-cluster and will fail with "
            f"'RPC listener disconnected'. Try: pip uninstall deltalake, then bump "
            f"FEATURE_GROUP_VERSION and re-run — or delete this feature group "
            f"version from the Hopsworks UI first if it already exists."
        )
    return fg


def apply_feature_descriptions(fg):
    """Best-effort — doesn't fail the pipeline if a column was renamed/removed."""
    for feature_name, description in FEATURE_DESCRIPTIONS.items():
        try:
            fg.update_feature_description(feature_name, description)
        except Exception as exc: 
            log.warning("Could not set description for %s: %s", feature_name, exc)


def insert_in_chunks(fg, df: pd.DataFrame):
    """Inserts the dataframe in chunks with retries. insert() is async by
    default in non-Spark clients, so we set wait_for_job=True per chunk to
    surface ingestion failures immediately rather than silently at the end."""
    n_chunks = (len(df) + INSERT_CHUNK_SIZE - 1) // INSERT_CHUNK_SIZE
    for i in range(n_chunks):
        chunk = df.iloc[i * INSERT_CHUNK_SIZE : (i + 1) * INSERT_CHUNK_SIZE]
        for attempt in range(1, MAX_INSERT_RETRIES + 1):
            try:
                log.info(
                    "Inserting chunk %d/%d (%d rows), attempt %d...",
                    i + 1, n_chunks, len(chunk), attempt,
                )
                fg.insert(chunk, write_options={"wait_for_job": True})
                break
            except Exception as exc:  
                wait = RETRY_BACKOFF_SECONDS * attempt
                log.warning(
                    "Chunk %d/%d failed (attempt %d/%d): %s — retrying in %ds",
                    i + 1, n_chunks, attempt, MAX_INSERT_RETRIES, exc, wait,
                )
                time.sleep(wait)
                if attempt == MAX_INSERT_RETRIES:
                    raise
    log.info("All %d rows inserted successfully.", len(df))


def main():
    df = load_features()
    fs = connect_feature_store()
    fg = get_or_create_fg(fs)
    apply_feature_descriptions(fg)
    insert_in_chunks(fg, df)


if __name__ == "__main__":
    main()

2026-08-21 12:10:17,023 [INFO] Loaded 17400 rows, 45 columns from faisalabad_aqi_2years.parquet
2026-08-21 12:10:17,025 [INFO] Closing external client and cleaning up certificates.
2026-08-21 12:10:17,027 [INFO] Connection closed.
2026-08-21 12:10:17,028 [INFO] Initializing external client
2026-08-21 12:10:17,029 [INFO] Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-08-21 12:10:20,728 [INFO] Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42119


2026-08-21 12:10:22,644 [INFO] Feature group 'weather_aqi_hourly' v5 resolved time_travel_format=HUDI
2026-08-21 12:10:22,645 [WARNING] Could not set description for city: "'FeatureGroup' object has no feature or transformation called 'city'."
2026-08-21 12:10:22,646 [WARNING] Could not set description for time: "'FeatureGroup' object has no feature or transformation called 'time'."
2026-08-21 12:10:22,646 [WARNING] Could not set description for us_aqi: "'FeatureGroup' object has no feature or transformation called 'us_aqi'."
2026-08-21 12:10:22,647 [WARNING] Could not set description for pm2_5: "'FeatureGroup' object has no feature or transformation called 'pm2_5'."
2026-08-21 12:10:22,647 [WARNING] Could not set description for pm10: "'FeatureGroup' object has no feature or transformation called 'pm10'."
2026-08-21 12:10:22,648 [WARNING] Could not set description for pm25_pm10_ratio: "'FeatureGroup' object has no feature or transformation called 'pm25_pm10_ratio'."
2026-08-21 12:10:2

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42119/fs/29780/fg/50911


Uploading Dataframe: 100.00% |██████████| Rows 3000/3000 | Elapsed Time: 00:05 | Remaining Time: 00:00


Launching job: weather_aqi_hourly_5_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/42119/jobs/named/weather_aqi_hourly_5_offline_fg_materialization/executions


2026-08-21 12:11:26,761 [INFO] Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-08-21 12:11:30,084 [INFO] Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-08-21 12:15:11,316 [INFO] Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-08-21 12:15:12,131 [INFO] Waiting for log aggregation to finish.
2026-08-21 12:15:12,132 [INFO] Execution finished successfully.
2026-08-21 12:15:12,133 [INFO] Inserting chunk 2/6 (3000 rows), attempt 1...
Uploading Dataframe: 100.00% |██████████| Rows 3000/3000 | Elapsed Time: 00:07 | Remaining Time: 00:00


Launching job: weather_aqi_hourly_5_offline_fg_materialization


2026-08-21 12:16:17,370 [WARNING] Chunk 2/6 failed (attempt 1/3): Remote end closed connection without response — retrying in 10s
2026-08-21 12:16:27,374 [INFO] Inserting chunk 2/6 (3000 rows), attempt 2...
Uploading Dataframe: 100.00% |██████████| Rows 3000/3000 | Elapsed Time: 00:05 | Remaining Time: 00:00
Use fg.materialization_job.run(args=-op offline_fg_materialization -path hdfs:///Projects/AQI_Predictor_fsd/Resources/jobs/weather_aqi_hourly_5_offline_fg_materialization/config_1787296223893) to trigger the materialization job again.
2026-08-21 12:16:40,674 [INFO] Inserting chunk 3/6 (3000 rows), attempt 1...
Uploading Dataframe: 100.00% |██████████| Rows 3000/3000 | Elapsed Time: 00:05 | Remaining Time: 00:00
Use fg.materialization_job.run(args=-op offline_fg_materialization -path hdfs:///Projects/AQI_Predictor_fsd/Resources/jobs/weather_aqi_hourly_5_offline_fg_materialization/config_1787296223893) to trigger the materialization job again.
2026-08-21 12:16:53,327 [INFO] Inserting

In [2]:
import hopsworks

project = hopsworks.login()
fs = project.get_feature_store(name='aqi_predictor_fsd_featurestore')
fg = fs.get_feature_group('weather_aqi_hourly', version=6) # or current version

# Force reading from online store
df = fg.read(online=True)
print(f"Data Verified! Retrieved {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.head())

2026-08-21 17:30:39,744 INFO: Closing external client and cleaning up certificates.
2026-08-21 17:30:39,746 INFO: Connection closed.
2026-08-21 17:30:39,748 INFO: Initializing external client
2026-08-21 17:30:39,749 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-21 17:30:43,027 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42119
Data Verified! Retrieved 17400 rows and 45 columns.
                 time  temperature_2m  relative_humidity_2m  surface_pressure  \
0 2024-08-25 00:00:00            27.4                    88             981.1   
1 2024-08-29 03:00:00            25.6                    87             982.3   
2 2024-08-29 10:00:00            26.9                    90             979.7   
3 2024-08-31 04:00:00            28.8                    79             985.5   
4 2024-08-24 00:00:00            26.5                    93             983.4   

   wind_speed_10m  wind_direction_10m  pm2_5  pm10  nitrogen_dioxide  ozone  \
0             6.0                 147   56.7  96.4              18.4   41.0   
1             9.5                  81   18.2  25.9              15.8   61.0   
2            10.4                  34   20.0  28.6               8.5  124.0   
3             4.1                 322   36.2  53.9         